In [1]:
from datetime import datetime, timedelta

import pandas as pd
import re
import imaplib
import email
from email.header import decode_header
from dotenv import load_dotenv
import os
from utils.get_emails import get_emails

from jmespath import search

In [48]:
log_file_path = "bulk-email-sender.log"
failed_emails = []
with open(log_file_path, 'r') as file:
    for line in file:
        if 'ERROR - Failed to send email to' in line:
            email = re.search(r'Failed to send email to (.*?):', line).group(1)
            failed_emails.append(email)
df_failed_emails_march_26 = pd.DataFrame(failed_emails, columns=['Failed Emails'])
print(df_failed_emails_march_26.head().to_markdown())

|    | Failed Emails                                           |
|---:|:--------------------------------------------------------|
|  0 | Kenya Commercial Bank (KCB)                             |
|  1 | info@equitybank.co.ke, customerservice@equitybank.co.ke |
|  2 | ceo@nockenya.co.ke and hr@nockenya.co.ke                |
|  3 | info@capitalradio.sd                                    |
|  4 | business.school@uofk.edu                                |


In [49]:
df_failed_emails_march_26["Failed Emails"] = df_failed_emails_march_26["Failed Emails"]
failed_emails_prospects = df_failed_emails_march_26["Failed Emails"].tolist()
print("Number of failed emails in pension: ", len(failed_emails_prospects))
print(failed_emails_prospects[:10])

Number of failed emails in pension:  77
['Kenya Commercial Bank (KCB)', 'info@equitybank.co.ke, customerservice@equitybank.co.ke', 'ceo@nockenya.co.ke and hr@nockenya.co.ke', 'info@capitalradio.sd', 'business.school@uofk.edu', 'info@royalcarehospital.com', 'anitak@flexi-personnel.com', 'admissions@brookhouse.ac.ke', 'info@datanetsd.com', 'admin@loretolimuru.co.ke']


In [ ]:
df_governance = pd.read_csv('emails/old/governance-email-cleaned.csv')
print("Number of records before dropping failed emails: ", len(df_governance))
df_governance_cleaned = df_governance.copy()
df_governance_cleaned = df_governance_cleaned[
    ~df_governance_cleaned['Emails'].str.strip().isin(failed_emails_prospects)]
print("Number of records after dropping failed emails: ", len(df_governance_cleaned))
print(df_governance_cleaned.head().to_markdown())

# Save the cleaned data to a new CSV file
df_governance_cleaned.to_csv('governance-email-super-cleaned.csv', index=False)
print("\nCleaned data saved to prospects-email-cleaned.csv")

In [ ]:
df_prospects = pd.read_csv('emails/old/prospects-email-cleaned.csv')
print("Number of records before dropping failed emails: ", len(df_prospects))
df_prospects_cleaned = df_prospects.copy()
df_prospects_cleaned = df_prospects_cleaned[~df_prospects_cleaned['Emails'].str.strip().isin(failed_emails_prospects)]
print("Number of records after dropping failed emails: ", len(df_prospects_cleaned))
print(df_prospects_cleaned.head().to_markdown())

# Save the cleaned data to a new CSV file
df_prospects_cleaned.to_csv('prospects-email-super-cleaned.csv', index=False)
print("\nCleaned data saved to prospects-email-cleaned.csv")

# March 26

## Filtering out failed emails

In [2]:
log_file_path = "bulk-email-sender.log"
failed_emails = []
with open(log_file_path, 'r') as file:
    for line in file:
        if 'ERROR - Failed to send email to' in line:
            email = re.search(r'Failed to send email to (.*?):', line).group(1)
            failed_emails.append(email)
df_failed_emails_march_26 = pd.DataFrame(failed_emails, columns=['Failed Emails'])
print("Total number of failed emails: ", len(failed_emails))
print(df_failed_emails_march_26.head().to_markdown())

Total number of failed emails:  77
|    | Failed Emails                                           |
|---:|:--------------------------------------------------------|
|  0 | Kenya Commercial Bank (KCB)                             |
|  1 | info@equitybank.co.ke, customerservice@equitybank.co.ke |
|  2 | ceo@nockenya.co.ke and hr@nockenya.co.ke                |
|  3 | info@capitalradio.sd                                    |
|  4 | business.school@uofk.edu                                |


## Fetching bounced emails

In [ ]:
def list_mailboxes(username, password):
    imap_server = "imap.gmail.com"
    mail = imaplib.IMAP4_SSL(imap_server)
    mailboxes = []

    try:
        # Login to the IMAP server
        mail.login(username, password)

        # Get the list of all available mailboxes
        status, mailbox_list = mail.list()
        if status != "OK":
            raise Exception("Failed to fetch mailbox list.")

        # Parse and format mailbox list
        for mailbox in mailbox_list:
            decoded_mailbox = mailbox.decode().split(' "/" ')[-1].strip('"')
            mailboxes.append(decoded_mailbox)

        return mailboxes
    except imaplib.IMAP4.error as imap_error:
        print(f"IMAP error occurred: {imap_error}")
        return []
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return []
    finally:
        try:
            mail.logout()
        except Exception as e:
            print(f"Error during logout: {e}")

In [99]:
import imaplib
import email
from email.header import decode_header
from datetime import datetime, timedelta

def get_failed_emails(username, password, start_date=None, end_date=None, max_emails=200):
    # Gmail IMAP server settings
    imap_server = "imap.gmail.com"

    # Create an IMAP4 client
    mail = imaplib.IMAP4_SSL(imap_server)

    mailbox_selected = False

    # Set default date range to yesterday if not provided
    if start_date is None:
        start_date = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0) - timedelta(days=1)

    if end_date is None:
        end_date = datetime.now().replace(hour=23, minute=59, second=59, microsecond=999999)

    # Convert date to the format IMAP expects (DD-MM-YYYY)
    start_date_str = start_date.strftime('%d-%b-%Y')
    end_date_str = end_date.strftime('%d-%b-%Y')

    try:
        # Log in to the account
        mail.login(username, password)

        # Select the specific folder
        status, _ = mail.select('"[Gmail]/Sent Mail"')
        if status != "OK":
            raise Exception('Unable to select mailbox. Ensure the folder name is correct.')

        mailbox_selected = True

        # Search for emails in the date range
        search_criteria = f'(SINCE {start_date_str} BEFORE {end_date_str})'
        status, search_data = mail.search(None, search_criteria)
        if status != "OK":
            raise Exception('Failed to search emails in the folder.')

        # Limit the number of emails to process
        email_ids = search_data[0].split()[:max_emails]

        failed_emails = []
        total_number_of_failed_emails = 0
        # Iterate through email IDs
        for num in email_ids:
            # Fetch full email content
            status, data = mail.fetch(num, '(RFC822)')
            print(f"Fetched email id: {num}, status: {status}")
            if status != "OK":
                print(f"Failed to fetch email with ID {num}. Skipping...")
                continue

            # Parse the email
            raw_email = data[0][1]
            email_message = email.message_from_bytes(raw_email)

            # Check for failure indicators
            failure_keywords = [
                'delivery failed',
                'delivery status notification',
                'undeliverable',
                'mailbox full',
                'quota exceeded',
                'host unknown',
                'recipient rejected',
                'permanent failure',
                'Address not found',
                "wasn't delivered",
            ]

            # Check email subject and body for failure indicators
            subject = email_message.get('Subject', '').lower()

            # Function to check email body for failure indicators
            def check_email_body(email_msg):
                # Handle multipart emails
                if email_msg.is_multipart():
                    for part in email_msg.walk():
                        content_type = part.get_content_type()
                        if content_type in ['text/plain', 'text/html']:
                            body = part.get_payload(decode=True).decode(errors='ignore').lower()
                            print(f"From muiltipart email body is {body}")
                            if any(keyword in body for keyword in failure_keywords):
                                return True
                else:
                    # Handle simple email
                    body = email_msg.get_payload(decode=True).decode(errors='ignore').lower()
                    print(f"From simple email body is {body}")
                    if any(keyword in body for keyword in failure_keywords):
                        return True
                return False

            # Check for failure indicators
            if (any(keyword in subject for keyword in failure_keywords) or
                check_email_body(email_message)):
                # Extract relevant information
                try:
                    # Try to extract original recipient
                    recipient = email_message.get('To', 'Unknown Recipient')
                    failed_emails.append(recipient)
                    total_number_of_failed_emails += 1
                    print(f"======Total number of failed emails: {failed_emails}=======")
                except Exception as e:
                    print(f"Error extracting email details: {e}")
            else:
                print(f"Email {num} does not contain any failure indicators.")

        return failed_emails

    except imaplib.IMAP4.error as imap_error:
        print(f"IMAP error occurred: {imap_error}")
        return []
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return []
    finally:
        # Safely close and logout
        try:
            if mailbox_selected:
                mail.close()
        except Exception as e:
            print(f"Error during closing mailbox: {e}")
        finally:
            try:
                mail.logout()
            except Exception as e:
                print(f"Error during logout: {e}")

In [100]:
def credentials_loader(env_path):
    os.environ.clear()
    load_dotenv(env_path)
    username = os.getenv("EMAIL_USERNAME")
    password = os.getenv("EMAIL_PASSWORD")
    return username, password

In [101]:
# Testing with it emails
it_env_path = ".env.it"
email_username, email_password = credentials_loader(it_env_path)
mail_list_folders = list_mailboxes(email_username, email_password)
print(mail_list_folders)

['INBOX', '[Gmail]', '[Gmail]/All Mail', '[Gmail]/Drafts', '[Gmail]/Important', '[Gmail]/Sent Mail', '[Gmail]/Spam', '[Gmail]/Starred', '[Gmail]/Trash']


In [102]:
# Testing with Pension emails
it_env_path = ".env.pension"
yesterday = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0) - timedelta(days=1)
start_date = yesterday.replace(hour=8, minute=43, second=0, microsecond=0)
end_date = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
print(yesterday)
print(end_date)
email_username, email_password = credentials_loader(it_env_path)

failed_pension_emails = get_failed_emails(email_username, email_password, yesterday, end_date)
len(failed_pension_emails)

2025-03-25 00:00:00
2025-03-26 00:00:00
Fetched email id: b'873', status: OK
From muiltipart email body is research indicates that organizations investing in team-building activities witness a 50% rise in employee engagement and a 35% improvement in overall productivity (forbes, 2024).

schools that prioritize bonding activities for staff and students also experience enhanced collaboration, improved teacher-student connections, and stronger retention rates.

teaching is a rewarding yet demanding profession that requires dedication, patience, and teamwork. ascent institute proposes a customized team building program for teachers and students to rejuvenate educators, strengthen relationships, and boost collaboration within your institution.

program objectives
- strengthen collaboration and communication among teachers and students.
- encourage creative problem-solving and critical thinking.
- foster a positive school culture and enhance morale.
- develop leadership skills and identify e

KeyboardInterrupt: 

In [68]:
failed_pension_emails[:10]

['nairobi.school@education.go.ke',
 'info@lenanaschool.sc.ke',
 'info@stareheboys.org',
 'info@kenyahigh.ac.ke',
 'pangani.girls@yahoo.com',
 'gigiri@braeburn.ac.ke',
 'admissions@brookhouse.ac.ke',
 'akpsnairobi@akdn.org',
 'office@peponischool.org',
 'admin@kenton.ac.ke']

In [70]:
# Testing with governance emails
it_env_path = ".env.pension"
email_username, email_password = credentials_loader(it_env_path)
failed_governance_emails = get_failed_emails(email_username, email_password)
len(failed_governance_emails)

(SINCE 25-Mar-2025 BEFORE 26-Mar-2025)
Fetched email id: b'873', status: OK
Fetched email id: b'874', status: OK
Fetched email id: b'875', status: OK
Fetched email id: b'876', status: OK
Fetched email id: b'877', status: OK
Fetched email id: b'878', status: OK
Fetched email id: b'879', status: OK
Fetched email id: b'880', status: OK
Fetched email id: b'881', status: OK
Fetched email id: b'882', status: OK
Fetched email id: b'883', status: OK
Fetched email id: b'884', status: OK
Fetched email id: b'885', status: OK
Fetched email id: b'886', status: OK
Fetched email id: b'887', status: OK
Fetched email id: b'888', status: OK
Fetched email id: b'889', status: OK
Fetched email id: b'890', status: OK
Fetched email id: b'891', status: OK
Fetched email id: b'892', status: OK
Fetched email id: b'893', status: OK
Fetched email id: b'894', status: OK
Fetched email id: b'895', status: OK
Fetched email id: b'896', status: OK
Fetched email id: b'897', status: OK
Fetched email id: b'898', status: OK

100

## April 24

In [51]:
log_file_path = "bulk-email-sender.log"
failed_emails = []
with open(log_file_path, 'r') as file:
    for line in file:
        if 'ERROR - Failed to send email to' in line:
            email = re.search(r'Failed to send email to (.*?):', line).group(1)
            failed_emails.append(email)
df_failed_emails_april_24 = pd.DataFrame(failed_emails, columns=['Failed Emails'])
print("Total number of failed emails: ", len(failed_emails))
print(df_failed_emails_april_24.head().to_markdown())

Total number of failed emails:  77
|    | Failed Emails                                           |
|---:|:--------------------------------------------------------|
|  0 | Kenya Commercial Bank (KCB)                             |
|  1 | info@equitybank.co.ke, customerservice@equitybank.co.ke |
|  2 | ceo@nockenya.co.ke and hr@nockenya.co.ke                |
|  3 | info@capitalradio.sd                                    |
|  4 | business.school@uofk.edu                                |


In [52]:
failed_emails_april_24 = df_failed_emails_april_24["Failed Emails"].tolist()
print("Number of failed emails in pension: ", len(failed_emails_april_24))
print(failed_emails_april_24[:10])

Number of failed emails in pension:  77
['Kenya Commercial Bank (KCB)', 'info@equitybank.co.ke, customerservice@equitybank.co.ke', 'ceo@nockenya.co.ke and hr@nockenya.co.ke', 'info@capitalradio.sd', 'business.school@uofk.edu', 'info@royalcarehospital.com', 'anitak@flexi-personnel.com', 'admissions@brookhouse.ac.ke', 'info@datanetsd.com', 'admin@loretolimuru.co.ke']


In [10]:
print(df_failed_emails_april_24.head().to_markdown())

|    | Failed Emails            |
|---:|:-------------------------|
|  0 | nan                      |
|  1 | info@equitybank.co.ke    |
|  2 | ceo@nockenya.co.ke       |
|  3 | info@capitalradio.sd     |
|  4 | business.school@uofk.edu |


In [53]:
df_uncleaned_full_emails_april_24 = pd.read_csv("emails/april/april-3/emails-final.csv")
df_uncleaned_full_emails_april_24.head()

,Emails,cc
0,gathoni.wathiga@brookside.co.ke,info@ascent-institute.com
1,food@dalgroup.com,info@ascent-institute.com
2,infozambia@tns.org,info@ascent-institute.com
3,nssshq@naivas.co.ke,info@ascent-institute.com
4,info@sudia.org,info@ascent-institute.com


In [54]:
print("Before dropping failed emails: ", len(df_uncleaned_full_emails_april_24))

df_cleaned_april_24 = df_uncleaned_full_emails_april_24[~(df_uncleaned_full_emails_april_24["Emails"].isin(failed_emails_april_24))]

print("After dropping failed emails: ", len(df_cleaned_april_24))

df_cleaned_april_24

Before dropping failed emails:  366
After dropping failed emails:  287


,Emails,cc
0,gathoni.wathiga@brookside.co.ke,info@ascent-institute.com
1,food@dalgroup.com,info@ascent-institute.com
2,infozambia@tns.org,info@ascent-institute.com
3,nssshq@naivas.co.ke,info@ascent-institute.com
4,info@sudia.org,info@ascent-institute.com
...,...,...
361,info@nctr.sd,info@ascent-institute.com
362,info@bidcoafrica.com,info@ascent-institute.com
363,cpsb@narok.go.ke,info@ascent-institute.com
364,jamleck.chege@sc.com,info@ascent-institute.com


In [55]:
df_cleaned_april_24.to_csv("emails/april/april-24/emails-cleaned.csv", index=False)

# April 25

In [3]:
log_file_path = "bulk-email-sender.log"
failed_emails = []
with open(log_file_path, 'r') as file:
    for line in file:
        if 'ERROR - Failed to send email to' in line:
            email = re.search(r'Failed to send email to (.*?):', line).group(1)
            failed_emails.append(email)
df_failed_emails_april_25 = pd.DataFrame(failed_emails, columns=['Failed Emails'])
print("Total number of failed emails: ", len(failed_emails))
print(df_failed_emails_april_25.head().to_markdown())

Total number of failed emails:  13
|    | Failed Emails                            |
|---:|:-----------------------------------------|
|  0 | compliance@mku.ac.ke, archives@mku.ac.ke |
|  1 | compliance@mku.ac.ke, archives@mku.ac.ke |
|  2 | info@sd.zain.com                         |
|  3 | kws@kws.go.ke                            |
|  4 | cpsb@nairobi.go.kez                      |


In [14]:
failed_emails_april_25 = df_failed_emails_april_25["Failed Emails"].tolist()
print("Number of failed emails in pension: ", len(failed_emails_april_25))

Number of failed emails in pension:  13


In [47]:
failed_it = pd.read_csv("emails/failed-emails/failed_emails-it.csv")["Failed Recipient"].str.strip().str.lower().tolist()
failed_governance = pd.read_csv("emails/failed-emails/failed_emails-governance.csv")["Failed Recipient"].str.strip().str.lower().tolist()
failed_pension = pd.read_csv("emails/failed-emails/failed_emails-pension.csv")["Failed Recipient"].str.strip().str.lower().tolist()

all_failed_emails = list(set(failed_it + failed_governance + failed_pension + failed_emails_april_25))
len(all_failed_emails)


123

In [48]:
# Emails from april 24
df_failed_emails_april_24 = pd.read_csv("emails/april/april-24/emails-cleaned.csv")
df_failed_emails_april_24["Emails"] = df_failed_emails_april_24["Emails"].str.strip().str.lower()
df_failed_emails_april_24.sample(30)

,Emails,cc
240,enquiries@braeburn.ac.ke,info@ascent-institute.com
75,monthlygiving@mercycorps.org,info@ascent-institute.com
35,hosp@nbihosp.org,info@ascent-institute.com
132,info@kenyahigh.ac.ke,info@ascent-institute.com
107,akpsnairobi@akdn.org,info@ascent-institute.com
177,jangoli@kenafricind.com,info@ascent-institute.com
64,info@kusco.org,info@ascent-institute.com
187,cpsb@kisii.go.ke,info@ascent-institute.com
198,cpsb@nyandarua.go.ke,info@ascent-institute.com
155,dvc@apd.jkuat.ac.ke,info@ascent-institute.com


In [56]:
all_failed_emails

['info@unitedcapitalbank.sd',
 'secretariat@zbidf.org.zm',
 'info@sudanairports.gov.sd',
 'contact@lycee-diderot.org',
 'info@arabsudaneseseed.com',
 'contactus@boakenya.com',
 'info@libertypensions.co.ke',
 'admin@kiambuhigh.sc.ke',
 'info@benefitsatwork.co.ke',
 'clinton@risivonneinvestments.com',
 'info@nairobi.go.ke',
 'info@bometassembly.go.ke',
 'nyonesaj12@yahoo.com',
 'info@rspoc.sd',
 'info@capitalradio.sd',
 'energy@dalgroup.com',
 'office@manguschool.sc.ke',
 'info@stareheboys.org',
 'vice-chancellor@kemu.ac.ke',
 'jnzisa@wauminisaacco.com',
 'archives@mku.ac.ke',
 'j.dimba@enwealth.co.ke',
 'customercare@natialbank.co.ke',
 'vicechancellor@kemu.ac.ke',
 'info@sedc.gov.sd',
 'info@alsudaninews.com',
 'dryassociatesipp@gmail.com',
 'hrsuppport@minet.co.ke',
 'office@loretolimuru.sc.ke',
 'info@mu.edu.zm',
 'info@modernarchitects.sd',
 'director.pa@pembe.o.ke',
 'cpsb@kisii.go.ke',
 'pangani.girls@yahoo.com',
 'info@danfodio.com',
 'infozambia@tns.org',
 'compliance@mku.ac.ke'

In [55]:
len(df_failed_emails_april_24[df_failed_emails_april_24["Emails"].isin(all_failed_emails)])

66

In [60]:
print("Before dropping failed emails: ", len(df_failed_emails_april_24))
df_cleaned_april_25 = df_failed_emails_april_24[~(df_failed_emails_april_24["Emails"] \
                                                  .isin(all_failed_emails))]
df_cleaned_april_25 = df_cleaned_april_25.drop(columns=['cc'])
df_cleaned_april_25 = df_cleaned_april_25.rename(columns={'Emails': 'Valid Emails'})
print("After dropping failed emails: ", len(df_cleaned_april_25))
df_cleaned_april_25.head()


Before dropping failed emails:  287
After dropping failed emails:  221


,Valid Emails
0,gathoni.wathiga@brookside.co.ke
1,food@dalgroup.com
3,nssshq@naivas.co.ke
4,info@sudia.org
7,callcentre@kra.go.ke


In [63]:
df_failed_emails_april_25 = pd.DataFrame(all_failed_emails, columns=['Failed Emails'])
df_failed_emails_april_25.drop_duplicates(inplace=True, subset=['Failed Emails'])
df_failed_emails_april_25.head()

,Failed Emails
0,info@unitedcapitalbank.sd
1,secretariat@zbidf.org.zm
2,info@sudanairports.gov.sd
3,contact@lycee-diderot.org
4,info@arabsudaneseseed.com


In [64]:
df_cleaned_april_25.to_csv("emails/april/april-25/emails-valid.csv", index=False)
df_failed_emails_april_25.to_csv("emails/april/april-25/emails-failed.csv", index=False)

# April 25

In [6]:
log_file_path = "bulk-email-sender.log"
failed_emails = []
with open(log_file_path, 'r') as file:
    for line in file:
        if 'ERROR - Failed to send email to' in line:
            email = re.search(r'Failed to send email to (.*?):', line).group(1)
            failed_emails.append(email)
print("Total number of failed emails: ", len(failed_emails))

Total number of failed emails:  13


In [ ]:
# Other failed emails
other_emails = [""]

In [3]:
df_failed_emails_april_25 = pd.read_csv("emails/april/april-25/emails-failed.csv")
failed_emails_april_25 = df_failed_emails_april_25["Failed Emails"].tolist()
failed_emails_april_25.extend(failed_emails)
df_all_emails = pd.read_csv("emails/april/april-25/emails-valid.csv")
df_all_emails_cleaned = df_all_emails[~(df_all_emails["Valid Emails"].isin(failed_emails_april_25))]


In [4]:
extracted_emails = get_emails(df_all_emails_cleaned["Valid Emails"].to_list())
df_cleaned_april_28 = pd.DataFrame(extracted_emails, columns=['Emails'])
df_cleaned_april_28["cc"] = "info@ascent-institute.com"
df_cleaned_april_28["Emails"] = df_cleaned_april_28["Emails"].str.strip().str.lower()
print("Before dropping duplicates: ", len(df_cleaned_april_28))
df_cleaned_april_28.drop_duplicates(inplace=True, subset=['Emails'])
print("After dropping duplicates: ", len(df_cleaned_april_28))
df_cleaned_april_28 = df_cleaned_april_28.rename(columns={'Emails': 'Valid Emails'})
df_cleaned_april_28.head()

Before dropping duplicates:  221
After dropping duplicates:  221


,Valid Emails,cc
0,gathoni.wathiga@brookside.co.ke,info@ascent-institute.com
1,food@dalgroup.com,info@ascent-institute.com
2,nssshq@naivas.co.ke,info@ascent-institute.com
3,info@sudia.org,info@ascent-institute.com
4,callcentre@kra.go.ke,info@ascent-institute.com


In [5]:
df_cleaned_april_28.to_csv("emails/april/april-28/emails-cleaned.csv", index=False)

# May 30

In [2]:
log_file_path = "bulk-email-sender.log"
failed_emails = []
with open(log_file_path, 'r') as file:
    for line in file:
        if 'ERROR - Failed to send email to' in line:
            email = re.search(r'Failed to send email to (.*?):', line).group(1)
            failed_emails.append(email)
print("Total number of failed emails: ", len(failed_emails))

Total number of failed emails:  148


In [3]:
failed_emails


['info@smartchampions.sacco.co.ke; smartchampionsacco@gmail.com',
 'cpsb@taitataveta.go.ke',
 'hr@jooust.ac.ke',
 'info@nafasisacco.co.ke',
 'rosemary.kyomugisha@rubisuganda.com',
 'cpsb@bungoma.go.ke',
 'info@mentorsacco.co.ke',
 'manan.desai@apainsurance.org',
 'info@strathmore.ac.ke',
 'kenyahigh@gmail.com',
 'info@kingdomsacco.com',
 'bosa@sheriasacco.coop',
 'jangoli@kenafricind.com',
 'info@transnationalsacco.co.ke',
 'sales@farmerschoice.co.ke',
 'chemelil.mdsoffice@gmail.com',
 'info@haggargroup.com',
 'monthlygiving@mercycorps.org',
 'cpsb@samburu.go.ke',
 'hr@aku.edu',
 'info@achievassacco.co.ke',
 'info@kenversitysacco.co.ke',
 'info@fedailhospital.com',
 'jscsecretariat@jsc.go.ke',
 'dimkessacco@yahoo.com',
 'info@garissa.go.ke',
 'info@shopperssacco.com',
 'sam.munda@travizory.com',
 'isaac.sigadah@haco.co.ke',
 'cpsb@turkana.go.ke',
 'paula.nyamwaya@kenya-airways.com',
 'info@kenpipesacco.com',
 'info@mudetesacco.co.ke',
 'countysec@mombasa.go.ke',
 'info@nacicosacco.coop

In [6]:
emails = get_emails(failed_emails)
len(emails)

111

In [7]:
df_failed_cleaned_emails = pd.DataFrame(emails, columns=['Emails'])
df_failed_cleaned_emails["cc"] = "info@ascent-institute.com"

In [9]:
df_failed_cleaned_emails.head()

,Emails,cc
0,cpsb@taitataveta.go.ke,info@ascent-institute.com
1,hr@jooust.ac.ke,info@ascent-institute.com
2,info@nafasisacco.co.ke,info@ascent-institute.com
3,rosemary.kyomugisha@rubisuganda.com,info@ascent-institute.com
4,cpsb@bungoma.go.ke,info@ascent-institute.com


In [10]:
df_failed_cleaned_emails.to_csv("emails/may/may-30/emails-failed-cleaned.csv", index=False)